# 16 - V2 validation tuning and central evaluation

Tune on validation only, freeze each dataset configuration, then evaluate the locked test once.


In [1]:
from pathlib import Path
import os, sys

LOCAL_CANDIDATES = [
    Path('/Volumes/LocalDrive1/onlyscholar-projects/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
]
DRIVE_CANDIDATES = [
    Path('/content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
    Path('/content/drive/MyDrive/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
    Path('/content/gdrive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
    Path('/content/gdrive/MyDrive/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
]

def is_project_root(path: Path) -> bool:
    return (path / 'pyproject.toml').is_file() and (path / 'src' / 'rareguard').is_dir()

def find_project_root() -> Path | None:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents, *LOCAL_CANDIDATES, *DRIVE_CANDIDATES]:
        if is_project_root(candidate):
            return candidate
    if Path('/content').is_dir():
        from google.colab import drive
        for mountpoint in (Path('/content/drive'), Path('/content/gdrive')):
            if mountpoint.is_dir() and any(mountpoint.iterdir()):
                continue
            try:
                drive.mount(str(mountpoint), force_remount=False)
            except ValueError as exc:
                if 'must not already contain files' not in str(exc):
                    raise
                continue
            for candidate in DRIVE_CANDIDATES:
                if is_project_root(candidate):
                    return candidate
    return None

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError('Canonical RareGuard-XIDS IMPLEMENTATION project not found.')
os.chdir(PROJECT_ROOT)
PROJECT_SRC = str(PROJECT_ROOT / 'src')
sys.path.insert(0, PROJECT_SRC)
os.environ['PYTHONPATH'] = PROJECT_SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
print('Project root:', PROJECT_ROOT)
print('Runtime:', 'Colab' if Path('/content').is_dir() else 'local')
print('Python path ready')


Mounted at /content/drive
Project root: /content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION
Runtime: Colab
Python path ready


In [2]:
DEVICE = 'cpu'
!python -m rareguard.experiments.run_publication_adaptive_tuning --config configs/publication_v2.yaml --datasets configs/datasets_v2.yaml --dataset all --device {DEVICE} --resume


2026-10-24 11:56:03,119 | INFO | rareguard | selected dataset=cse_cic_ids2018 validation_score=0.6429 settings=(0.500, 0.250, 0.100)
2026-10-24 11:56:05,808 | INFO | rareguard | selected dataset=cic_bccc_nrc_tabulariotattack_2024 validation_score=0.3698 settings=(0.000, 0.000, 0.100)
2026-10-24 11:56:07,521 | INFO | rareguard | selected dataset=cic_iiot_2025_datasense validation_score=0.8096 settings=(0.000, 0.000, 0.100)
2026-10-24 11:56:08,601 | INFO | rareguard | selected dataset=edge_iiotset validation_score=0.8954 settings=(0.000, 0.000, 0.100)
2026-10-24 11:56:09,289 | INFO | rareguard | frozen confirmation dataset=ciciot2023 from development-only validation score=0.6692 settings=(0.000, 0.000, 0.100)
2026-10-24 11:56:09,535 | INFO | rareguard | frozen confirmation dataset=ton_iot from development-only validation score=0.6692 settings=(0.000, 0.000, 0.100)


In [3]:
!python -m rareguard.experiments.run_publication_baselines --config configs/publication_v2.yaml --datasets configs/datasets_v2.yaml --dataset all --resume


In [4]:
import subprocess
import torch

DEVELOPMENT_DATASETS = [
    'cse_cic_ids2018',
    'edge_iiotset',
    'cic_bccc_nrc_tabulariotattack_2024',
    'cic_iiot_2025_datasense',
]
CONFIRMATION_DATASETS = ['ciciot2023', 'ton_iot']

if not torch.cuda.is_available():
    raise RuntimeError('A CUDA runtime is required for the development-dataset stage.')

def run_device_split_stage(module: str) -> None:
    for dataset in DEVELOPMENT_DATASETS:
        subprocess.run(
            [sys.executable, '-m', module, '--config', 'configs/publication_v2.yaml',
             '--datasets', 'configs/datasets_v2.yaml', '--dataset', dataset,
             '--device', 'cuda', '--resume'],
            check=True,
        )
    for dataset in CONFIRMATION_DATASETS:
        subprocess.run(
            [sys.executable, '-m', module, '--config', 'configs/publication_v2.yaml',
             '--datasets', 'configs/datasets_v2.yaml', '--dataset', dataset,
             '--device', 'cpu', '--resume'],
            check=True,
        )

run_device_split_stage('rareguard.experiments.run_publication_central')


In [5]:
import pandas as pd
display(pd.read_csv('artifacts/publication_v2/results/central_summary_ci.csv'))


,dataset,model,seeds,best_epoch_mean,best_epoch_ci95_half,best_validation_selection_score_mean,best_validation_selection_score_ci95_half,best_validation_f1_macro_mean,best_validation_f1_macro_ci95_half,best_validation_rare_f1_macro_mean,...,ece_15_mean,ece_15_ci95_half,rare_class_count_mean,rare_class_count_ci95_half,rare_f1_macro_mean,rare_f1_macro_ci95_half,roc_auc_macro_ovr_mean,roc_auc_macro_ovr_ci95_half,pr_auc_macro_mean,pr_auc_macro_ci95_half
0,cic_bccc_nrc_tabulariotattack_2024,NOVA-XIDS,5,20.400000,5.133554,0.344697,0.025523,0.562044,0.035230,0.127350,...,0.038735,0.009378,1.0,0.0,0.122153,0.026131,0.936392,0.005385,0.594154,0.024531
1,cic_bccc_nrc_tabulariotattack_2024,RareGuard-XIDS,5,10.600000,4.903918,0.369699,0.009980,0.595787,0.009142,0.143611,...,0.021800,0.003196,1.0,0.0,0.123908,0.031231,0.941993,0.002699,0.617661,0.013463
2,cic_bccc_nrc_tabulariotattack_2024,RareGuard-no-class-balance,3,23.000000,1.131607,0.307576,0.001787,0.615152,0.003574,0.000000,...,0.015551,0.003105,1.0,0.0,0.000000,0.000000,0.949131,0.001365,0.642150,0.001877
3,cic_bccc_nrc_tabulariotattack_2024,RareGuard-no-contrastive,3,17.000000,6.300518,0.385452,0.013672,0.606494,0.012023,0.164411,...,0.021258,0.002519,1.0,0.0,0.132249,0.002125,0.944778,0.002367,0.630559,0.012133
4,cic_bccc_nrc_tabulariotattack_2024,RareGuard-no-direct-path,3,17.666667,7.534608,0.371792,0.033192,0.593827,0.033678,0.149757,...,0.020205,0.003926,1.0,0.0,0.111662,0.062901,0.940769,0.007378,0.617130,0.020428
5,cic_bccc_nrc_tabulariotattack_2024,RareGuard-no-prototype,3,10.666667,9.216401,0.371751,0.021595,0.588856,0.026330,0.154645,...,0.021757,0.009241,1.0,0.0,0.145268,0.048712,0.941794,0.005621,0.614670,0.023797
6,cic_iiot_2025_datasense,NOVA-XIDS,5,22.200000,0.733365,0.352856,0.030745,0.335451,0.026441,0.370261,...,0.118586,0.025022,18.0,0.0,0.361634,0.036063,0.969394,0.002283,0.666384,0.023092
7,cic_iiot_2025_datasense,RareGuard-XIDS,5,19.400000,4.045394,0.807395,0.015331,0.804580,0.012436,0.810211,...,0.029937,0.003157,18.0,0.0,0.817613,0.021829,0.993788,0.000527,0.881171,0.007479
8,cic_iiot_2025_datasense,RareGuard-no-class-balance,3,23.000000,1.131607,0.820542,0.008481,0.821372,0.005838,0.819711,...,0.017451,0.004254,18.0,0.0,0.842973,0.007765,0.994325,0.000567,0.888717,0.007611
9,cic_iiot_2025_datasense,RareGuard-no-contrastive,3,22.666667,2.613333,0.814188,0.017123,0.810010,0.009199,0.818366,...,0.032829,0.007377,18.0,0.0,0.812031,0.025155,0.994353,0.000146,0.881852,0.015173
